# Finding similar items
The aim is to detect similar textual items in the `text` field of the Kaggle [Yelp](https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset) dataset.

In [ ]:
import os
import json
import pandas as pd
import pip
import string

def import_or_install(package):
    try:
        __import__(package)
    except ImportError:
        pip.main(['install', package])

foldername = "/content/yelp-dataset"

In [2]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

import findspark
findspark.init("spark-3.5.0-bin-hadoop3")# SPARK_HOME
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

sc = spark.sparkContext

We first of all import the Yelp dataset from Kaggle

In [3]:
os.environ['KAGGLE_USERNAME'] = "xxx"
os.environ['KAGGLE_KEY'] = "xxx"
!kaggle datasets download -d yelp-dataset/yelp-dataset

100% 4.07G/4.07G [00:43<00:00, 145MB/s]
100% 4.07G/4.07G [00:43<00:00, 100MB/s]


In [6]:
from tqdm import tqdm
import zipfile
with zipfile.ZipFile(foldername + ".zip","r") as zip_ref:
     for file in tqdm(iterable=zip_ref.namelist(), total=len(zip_ref.namelist())):
          zip_ref.extract(member=file)

100%|██████████| 6/6 [02:03<00:00, 20.61s/it]


We take into consideration the portion of the dataset regarding reviews, contained in `yelp_academic_dataset_review.json`, and we keep the resulting dataframe in RDD form.

In [7]:
df_reviews = spark.read.json(foldername + "/yelp_academic_dataset_review.json")
rdd_reviews = df_reviews.rdd

Seen the aim of the project we will only be looking at the `text` attribute of the imported reviews.

In [29]:
rdd_reviews_text = rdd_reviews.map(lambda x: x['text'])

## Data pre-processing

We begin by removing text cells that are `None` or that contain empty strings. It is possible to verify that for each review a nonempty text is present.

In [30]:
rdd_reviews_text = rdd_reviews_text.filter(lambda text: bool(text))

To arrive at a list of significant words for the text of each review we apply the following modifications:
* remove all punctuation and control characters,
* set all characters to lowercase,
* obtain all words from the text paragraph,
* lemmatize,
* remove all stop words,
* maintain each word only once.

In [31]:
%%capture
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
import spacy
lemmatizer = spacy.load("en_core_web_sm")

#list of transfomations to apply to the set of review texts
remove_punctuation = lambda text: text.translate(str.maketrans('', '', string.punctuation+"\n"))
text_to_lowercase = lambda text: text.lower()
lemmatize = lambda text: " ".join([token.lemma_ for token in lemmatizer(text)])
text_to_words = lambda text: text.split(" ")
remove_stopwords = lambda wordlist: [word for word in wordlist if word not in stopwords.words('english')]
remove_duplicates = lambda wordlist: list(set(wordlist))
preprocessing = [remove_punctuation, text_to_lowercase, lemmatize, text_to_words, remove_stopwords, remove_duplicates]

for func in preprocessing:
    rdd_reviews_text = rdd_reviews_text.map(func)